In [1]:
import pandas as pd;
import numpy as np;


columns_of_interest = ['OBJECTID','DISCOVERY_DATE', 'DISCOVERY_TIME', 'NWCG_GENERAL_CAUSE', 'CONT_DATE', 'CONT_TIME', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE' , 'LONGITUDE' , 'STATE' ]

renamed_columns = ["object_id", "discovery_date", "discovery_time", "general_cause", "controlled_date", "controlled_time", "fire_size", "fire_class", "latitude", "longitude", "state", "county"]

map_cause = {'Power generation/transmission/distribution':'Accidental',
            'Natural':'Natural',
            'Debris and open burning':'Accidental',
            'Missing data/not specified/undetermined':'Undefined',
            'Recreation and ceremony':'Accidental',
            'Equipment and vehicle use':'Accidental',
            'Arson/incendiarism':'Criminal',
            'Fireworks':'Accidental',
            'Other causes':'Accidental',
            'Railroad operations and maintenance':'Accidental',
            'Smoking':'Accidental',
            'Misuse of fire by a minor':'Accidental',
            'Firearms and explosives use':'Accidental'}

# DOWNLOAD FILE FIRST: https://syd1.digitaloceanspaces.com/duckgoesmeow/bushfire-data/data.csv
FILE_PATH = "~/Desktop/archive/data.csv"
bushfire_df = pd.read_csv(FILE_PATH, low_memory=False)

# bushfire_df = bushfire_df[columns_of_interest].set_index()

# retreive the first n rows from top to bottom
# bushfire_df.head(30)

# selected columns - retreive first n rows from top to bottom
# bushfire_df[columns_of_interest].head(30)


# retreive last n rows from bottom to top
# bushfire_df.tail(50)

# selected columns - retreive last n rows from top to bottom
# bushfire_df[columns_of_interest].tail(30)

# retreive random sample of data - specify size of rows
# bushfire_df.sample(30)

# selected columns - retreive random sample of data - specify size of rows
# bushfire_df[columns_of_interest].sample(30)


# Let's grab specific rows with specific data types:
# bushfire_df[columns_of_interest].sample(10).loc[:].isna()

# What if we wanted to select a column and get row data which is of specific type...
# With spread syntax:
# bushfire_df[columns_of_interest].loc[(bushfire_df['DISCOVERY_TIME'].isna() == True, [*columns_of_interest])] 

# or pick certain columns to display with successful condition:
# bushfire_df[columns_of_interest].loc[(bushfire_df['DISCOVERY_TIME'].isna() == True), ['STATE', 'DISCOVERY_DATE', 'DISCOVERY_TIME', 'LONGITUDE', 'LATITUDE', 'NWCG_GENERAL_CAUSE']]

# Let's rename the columns:
bushfire_df = bushfire_df.rename(columns={
                         'OBJECTID':'object_id',
                         'DISCOVERY_DATE':'discovery_date',
                         'DISCOVERY_TIME' : 'discovery_time',
                         'NWCG_GENERAL_CAUSE':'general_cause',
                         'CONT_DATE':'controlled_date',
                         'CONT_TIME':'controlled_time',
                         'FIRE_SIZE' : 'fire_size',
                         'FIRE_SIZE_CLASS' : 'fire_class',
                         'LATITUDE': 'latitude',
                         'LONGITUDE':'longitude',
                         'COUNTY' : 'county',
                         'STATE':'state'}).rename_axis('index_id')




In [2]:
bushfire_df = bushfire_df[renamed_columns]

In [3]:
bushfire_df['discovery_date'] = pd.to_datetime(arg=bushfire_df['discovery_date']).astype(str)

In [4]:
bushfire_df['discovery_time'] = pd.to_numeric(bushfire_df['discovery_time'], errors='coerce').fillna(0).astype(int).astype(str)

In [5]:
def format_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).zfill(4)
    if time_str == '0000' or time_str == '2400':
        return '00:00'
    elif 0 <= int(time_str) <= 2359:
        return f"{time_str[:2]}:{time_str[2:]}"
    else: 
        return np.nan
    

bushfire_df['discovery_time'] = bushfire_df['discovery_time'].apply(format_time)

In [6]:
bushfire_df['discovery_datetime'] = bushfire_df['discovery_date'] + " " + bushfire_df['discovery_time']

In [7]:
bushfire_df['discovery_datetime'] = pd.to_datetime(arg=bushfire_df['discovery_datetime'], format='%Y-%m-%d %H:%M')

In [8]:
bushfire_df.insert(11, 'discovery_day', bushfire_df['discovery_datetime'].dt.day_name())

In [9]:
bushfire_df['discovery_datetime'] = bushfire_df['discovery_datetime'].dt.strftime('%Y-%m-%d %H:%M')

In [10]:
bushfire_df.drop(columns=['discovery_date', 'discovery_time'], axis=1, inplace=True)

In [11]:
bushfire_df.drop(columns=['county'], axis=1, inplace=True)

In [12]:
map_cause = {'Power generation/transmission/distribution':'Accidental',
            'Natural':'Natural',
            'Debris and open burning':'Accidental',
            'Missing data/not specified/undetermined':'Unidentified',
            'Recreation and ceremony':'Accidental',
            'Equipment and vehicle use':'Accidental',
            'Arson/incendiarism':'Criminal',
            'Fireworks':'Accidental',
            'Other causes':'Accidental',
            'Railroad operations and maintenance':'Accidental',
            'Smoking':'Accidental',
            'Misuse of fire by a minor':'Accidental',
            'Firearms and explosives use':'Accidental'}

bushfire_df['origin'] = bushfire_df['general_cause'].map(map_cause)


In [13]:
cols_to_fill = ['controlled_date', 'controlled_time']
bushfire_df[cols_to_fill] = bushfire_df[cols_to_fill].fillna('unknown')

In [14]:
bushfire_df.sample(20)

,object_id,general_cause,controlled_date,controlled_time,fire_size,fire_class,latitude,longitude,state,discovery_day,discovery_datetime,origin
index_id,,,,,,,,,,,,
893078,893079,Missing data/not specified/undetermined,unknown,unknown,0.10,A,32.996900,-93.713600,LA,Friday,2000-01-07 00:00,Unidentified
1864308,1864309,Missing data/not specified/undetermined,unknown,unknown,0.10,A,21.324673,-158.084641,HI,Monday,2005-04-18 00:00,Unidentified
2176767,2176768,Missing data/not specified/undetermined,8/11/2019,720.0,2.00,B,39.486900,-119.993600,NV,Saturday,2019-08-10 19:39,Unidentified
1331780,1331781,Debris and open burning,8/29/2004,1450.0,0.10,A,41.412251,-74.726977,NY,Sunday,2004-08-29 14:50,Accidental
1581258,1581259,Debris and open burning,7/14/2012,1430.0,0.20,A,42.970800,-108.497800,WY,Saturday,2012-07-14 14:05,Accidental
523065,523066,Missing data/not specified/undetermined,unknown,unknown,0.20,A,35.677140,-79.422740,NC,Friday,2007-08-03 00:00,Unidentified
434435,434436,Railroad operations and maintenance,unknown,unknown,0.62,B,31.440634,-82.068240,GA,Sunday,2004-05-09 00:00,Accidental
1689257,1689258,Missing data/not specified/undetermined,unknown,unknown,0.50,B,33.068074,-97.313839,TX,Wednesday,2013-02-27 00:00,Unidentified
1967212,1967213,Missing data/not specified/undetermined,8/19/2016,unknown,0.25,A,41.910225,-70.735740,MA,Friday,2016-08-19 07:48,Unidentified


In [15]:
output_file = './cleaned-bushfire-data.csv'
bushfire_df.to_csv(output_file, index=False, encoding='utf-8')


In [16]:
cleaned_df = pd.read_csv('./cleaned-bushfire-data.csv')


In [17]:
cleaned_df.sample(20)

,object_id,general_cause,controlled_date,controlled_time,fire_size,fire_class,latitude,longitude,state,discovery_day,discovery_datetime,origin
909308,909309,Missing data/not specified/undetermined,unknown,unknown,40.00,C,33.615800,-81.085000,SC,Sunday,1998-06-14 00:00,Unidentified
655004,655005,Debris and open burning,11/24/2008,1934.0,2.00,B,32.007690,-95.185250,TX,Monday,2008-11-24 18:00,Accidental
1077609,1077610,Missing data/not specified/undetermined,unknown,unknown,1.00,B,37.350000,-120.576944,CA,Sunday,2009-03-29 19:23,Unidentified
2204477,2204478,Recreation and ceremony,11/19/2019,1715.0,1.20,B,36.288050,-87.540417,TN,Tuesday,2019-11-19 15:55,Accidental
650220,650221,Debris and open burning,unknown,unknown,5.00,B,30.355890,-94.295220,TX,Wednesday,2008-03-12 00:00,Accidental
5644,5645,Natural,6/26/2005,1303.0,0.10,A,35.061667,-112.465278,AZ,Sunday,2005-06-26 07:33,Natural
79953,79954,Arson/incendiarism,10/9/1995,300.0,0.30,B,34.171667,-117.670000,CA,Sunday,1995-10-08 23:45,Criminal
1409055,1409056,Debris and open burning,4/14/1995,1630.0,0.20,A,34.653000,-85.071700,GA,Friday,1995-04-14 15:00,Accidental
1780754,1780755,Missing data/not specified/undetermined,unknown,unknown,0.10,A,41.030500,-73.772100,NY,Wednesday,2014-06-25 16:53,Unidentified
1967163,1967164,Missing data/not specified/undetermined,8/18/2016,1715.0,0.01,A,39.868600,-108.731900,CO,Thursday,2016-08-18 17:15,Unidentified
